<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip__Interativo_V2_BARCHART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BIBLIOTECA

In [43]:
# ================================================================================
# ### BIBLIOTECA
# ================================================================================


# CÉLULA 1
### Rodar essa célula somente uma vez ###
# Delete a # na linha abaixo, execute e coloque de volta a #

#!pip install plotly


# CÉLULA 2
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date


# CÉLULA 3
pd.options.display.float_format = '{:,.4f}'.format

### ARQUIVO CSV

In [44]:
# ================================================================================
# ### ARQUIVO CSV
# ================================================================================


# CÉLULA 5
# Parametros de entrada
filename = 'volatility-greeks-exp.csv'


In [45]:
# ==============================================================================
# BLOCO DE ADAPTAÇÃO PARA O NOVO CSV
# ==============================================================================
# Carregar e limpar o novo CSV
df_new = pd.read_csv(filename, skipfooter=1, engine='python')
df_new.dropna(subset=['Type'], inplace=True)

# Limpeza dos dados
if df_new['Strike'].dtype == 'object':
    df_new['Strike'] = df_new['Strike'].str.replace(',', '', regex=False).astype(float)
for col in ['IV', 'IV Skew']:
    if col in df_new.columns and df_new[col].dtype == 'object':
        df_new[col] = df_new[col].str.replace('%', '', regex=False).astype(float) / 100.0
for col in ['Last', 'Delta', 'Gamma', 'Theta', 'Vega']:
    if col in df_new.columns:
        df_new[col] = pd.to_numeric(df_new[col], errors='coerce')
df_new.dropna(inplace=True)

# Estimar spotPrice
temp_calls = df_new[df_new['Type'] == 'Call'][['Strike', 'Last']].set_index('Strike')
temp_puts = df_new[df_new['Type'] == 'Put'][['Strike', 'Last']].set_index('Strike')
merged_prices = pd.merge(temp_calls, temp_puts, left_index=True, right_index=True, how='inner', suffixes=('_call', '_put'))
merged_prices['diff'] = abs(merged_prices['Last_call'] - merged_prices['Last_put'])
spotPrice = merged_prices['diff'].idxmin() if not merged_prices.empty else (df_new['Strike'].max() + df_new['Strike'].min()) / 2
fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice
todayDate = datetime.now()

# Adicionar colunas faltantes
if 'OpenInt' not in df_new.columns: df_new['OpenInt'] = 1
if 'Vol' not in df_new.columns: df_new['Vol'] = 0

# Pivotar para formato original
calls_df = df_new[df_new['Type'] == 'Call'].set_index('Strike')
puts_df = df_new[df_new['Type'] == 'Put'].set_index('Strike')
df = pd.merge(calls_df, puts_df, left_index=True, right_index=True, how='outer', suffixes=('_Call', '_Put'))
df.reset_index(inplace=True)

# Renomear colunas para o formato original
df.rename(columns={
    'Strike': 'StrikePrice',
    'Gamma_Call': 'CallGamma',
    'OpenInt_Call': 'CallOpenInt',
    'Delta_Call': 'CallDelta',
    'Vol_Call': 'CallVol',
    'Last_Call': 'CallLastSale',
    'IV_Call': 'CallIV',
    'Gamma_Put': 'PutGamma',
    'OpenInt_Put': 'PutOpenInt',
    'Delta_Put': 'PutDelta',
    'Vol_Put': 'PutVol',
    'Last_Put': 'PutLastSale',
    'IV_Put': 'PutIV'
}, inplace=True)

# Adicionar colunas do formato original
for col in ['ExpirationDate','Calls','CallLastSale','CallNet','CallBid','CallAsk','CallVol',
            'CallIV','CallDelta','CallGamma','CallOpenInt','StrikePrice','Puts','PutLastSale',
            'PutNet','PutBid','PutAsk','PutVol','PutIV','PutDelta','PutGamma','PutOpenInt']:
    if col not in df.columns:
        df[col] = 0

df.fillna(0, inplace=True)
df['ExpirationDate'] = pd.to_datetime("2025-12-19") + timedelta(hours=16)
df['StrikePrice'] = df['StrikePrice'].astype(float)
df['CallIV'] = df['CallIV'].astype(float)
df['PutIV'] = df['PutIV'].astype(float)
df['CallGamma'] = df['CallGamma'].astype(float)
df['PutGamma'] = df['PutGamma'].astype(float)
df['CallOpenInt'] = df['CallOpenInt'].astype(float)
df['PutOpenInt'] = df['PutOpenInt'].astype(float)
# FIM DA ADAPTAÇÃO
# ==============================================================================

### GAMMA

In [46]:
# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else: # Gamma is same for calls and puts. This is just to cross-check
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [47]:
# ================================================================================
# ### GAMMA (CHART 1, 2 E 3)
# ================================================================================


# CÉLULA 13
# ---=== CALCULATE SPOT GAMMA ===---
# Gamma Exposure = Unit Gamma * Open Interest * Contract Size * Spot Price
# To further convert into 'per 1% move' quantity, multiply by 1% of spotPrice
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values


# CÉLULA 14
# Chart 1: Absolute Gamma Exposure
# define os dados
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

# cria um gráfico de barras
fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Gamma Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig.show()



In [48]:
# CÉLULA 15
# CALL WALL E PUT WALL - ALL DTE
call_oi_wall_strike = df.loc[df['CallOpenInt'].idxmax()]['StrikePrice']
call_oi_wall_value = df['CallOpenInt'].max()

# Find the strike with the highest Put Open Interest and its value
put_oi_wall_strike = df.loc[df['PutOpenInt'].idxmax()]['StrikePrice']
put_oi_wall_value = df['PutOpenInt'].max()

# Find the strike with the highest Call Volume and its value
call_vol_wall_strike = df.loc[df['CallVol'].idxmax()]['StrikePrice']
call_vol_wall_value = df['CallVol'].max()

# Find the strike with the highest Put Volume and its value
put_vol_wall_strike = df.loc[df['PutVol'].idxmax()]['StrikePrice']
put_vol_wall_value = df['PutVol'].max()

# Find the top 5 strikes with the highest Call Open Interest
top5_call_oi = df.nlargest(5, 'CallOpenInt')[['StrikePrice', 'CallOpenInt']]

# Find the top 5 strikes with the highest Put Open Interest
top5_put_oi = df.nlargest(5, 'PutOpenInt')[['StrikePrice', 'PutOpenInt']]


print("--- Dados de Open Interest e Volume para Call/Put Walls ---")

print("\nParedes por Open Interest:")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")

print("\nParedes por Volume:")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\nTop 5 Calls por Open Interest:")
for index, row in top5_call_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")

print("\nTop 5 Puts por Open Interest:")
for index, row in top5_put_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")

--- Dados de Open Interest e Volume para Call/Put Walls ---

Paredes por Open Interest:
  Call Wall (OI): 1 at strike 24925
  Put Wall (OI): 1 at strike 24925

Paredes por Volume:
  Call Wall (Vol): 0 at strike 24925
  Put Wall (Vol): 0 at strike 24925

Top 5 Calls por Open Interest:
  Strike 24925: 1
  Strike 24950: 1
  Strike 24975: 1
  Strike 25000: 1
  Strike 25025: 1

Top 5 Puts por Open Interest:
  Strike 24925: 1
  Strike 24950: 1
  Strike 24975: 1
  Strike 25000: 1
  Strike 25025: 1


In [49]:
# CÉLULA 16
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the top 9 smallest gamma values (most negative)
smallest_gamma = dfAgg_sorted.head(9)
print("Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
# Iterate through smallest_gamma in descending order of TotalGamma (already sorted ascending, so reverse)
for index, row in smallest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Get the top 9 largest gamma values (most positive)
largest_gamma = dfAgg_sorted.tail(9)
print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
# Iterate through largest_gamma in descending order of TotalGamma (already sorted ascending, so reverse)
for index, row in largest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Add the Zero Gamma (Gamma Flip) point
print(f"\nZero Gamma (Gamma Flip): {zeroGamma:.0f}")

# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")


# --- Find the single largest strike by Volume and Open Interest (Overall) ---

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
largest_call_vol_strike = df.loc[df['CallVol'].idxmax()]
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
largest_put_vol_strike = df.loc[df['PutVol'].idxmax()]
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
largest_call_oi_strike = df.loc[df['CallOpenInt'].idxmax()]
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
largest_put_oi_strike = df.loc[df['PutOpenInt'].idxmax()]
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")


Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):
  Strike 25150: Type: P, Delta: 2.3507 M, Gamma: -0.0001 Bn
  Strike 25025: Type: P, Delta: 2.4002 M, Gamma: -0.0001 Bn
  Strike 25050: Type: P, Delta: 2.3872 M, Gamma: -0.0001 Bn
  Strike 25075: Type: P, Delta: 2.3769 M, Gamma: -0.0001 Bn
  Strike 25250: Type: P, Delta: 2.1064 M, Gamma: -0.0001 Bn
  Strike 25100: Type: P, Delta: 2.3618 M, Gamma: -0.0001 Bn
  Strike 25225: Type: P, Delta: 2.1433 M, Gamma: -0.0001 Bn
  Strike 25200: Type: P, Delta: 2.1316 M, Gamma: -0.0001 Bn
  Strike 25125: Type: P, Delta: 2.2241 M, Gamma: -0.0001 Bn

Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):
  Strike 26375: Type: C, Delta: -0.4339 M, Gamma: 0.0003 Bn
  Strike 26075: Type: C, Delta: 0.4615 M, Gamma: 0.0002 Bn
  Strike 26450: Type: C, Delta: -0.5824 M, Gamma: 0.0002 Bn
  Strike 25950: Type: C, Delta: 1.2336 M, Gamma: 0.0002 Bn
  Strike 26050: Type: C, Delta: 0.6861 M, Gamma: 0.0002 Bn
  Strike 26350: Type: C, Delta: -0.3

In [50]:
# CÉLULA 17
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])
chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")
fig.add_shape(dict(type="line", x0=spotPrice, y0=0, x1=spotPrice, y1=max(dfAgg['CallGEX'].to_numpy() / 10**9), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)

fig.show()


In [51]:
# CÉLULA 18
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")


Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 26050, Call GEX: 0.0006 Bn, Put GEX: -0.0004 Bn
  GEX Level 2: Strike 26225, Call GEX: 0.0006 Bn, Put GEX: -0.0004 Bn
  GEX Level 3: Strike 25950, Call GEX: 0.0006 Bn, Put GEX: -0.0004 Bn
  GEX Level 4: Strike 25975, Call GEX: 0.0005 Bn, Put GEX: -0.0004 Bn
  GEX Level 5: Strike 26025, Call GEX: 0.0006 Bn, Put GEX: -0.0004 Bn
  GEX Level 6: Strike 26425, Call GEX: 0.0005 Bn, Put GEX: -0.0004 Bn


In [52]:
# CÉLULA 19
# ---=== CALCULATE GAMMA PROFILE ===---
levels = np.linspace(fromStrike, toStrike, 60)

# For 0DTE options, I'm setting DTE = 1 day, otherwise they get excluded
df['daysTillExp'] = [1/262 if (np.busday_count(todayDate.date(), x.date())) == 0 \
                           else np.busday_count(todayDate.date(), x.date())/262 for x in df.ExpirationDate]

nextExpiry = df['ExpirationDate'].min()

df['IsThirdFriday'] = [isThirdFriday(x) for x in df.ExpirationDate]
thirdFridays = df.loc[df['IsThirdFriday'] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min()

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

In [53]:
# CÉLULA 20
# For each spot level, calc gamma exposure at that point
for level in levels:
    df['callGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df['putGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma.append(df['callGammaEx'].sum() - df['putGammaEx'].sum())

    exNxt = df.loc[df['ExpirationDate'] != nextExpiry]
    totalGammaExNext.append(exNxt['callGammaEx'].sum() - exNxt['putGammaEx'].sum())

    exFri = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalGammaExFri.append(exFri['callGammaEx'].sum() - exFri['putGammaEx'].sum())

totalGamma = np.array(totalGamma) / 10**9
totalGammaExNext = np.array(totalGammaExNext) / 10**9
totalGammaExFri = np.array(totalGammaExFri) / 10**9

In [54]:
# CÉLULA 21
# Find Gamma Flip Point
zeroCrossIdx = np.where(np.diff(np.sign(totalGamma)))[0]

negGamma = totalGamma[zeroCrossIdx]
posGamma = totalGamma[zeroCrossIdx+1]
negStrike = levels[zeroCrossIdx]
posStrike = levels[zeroCrossIdx+1]

zeroGamma = posStrike - ((posStrike - negStrike) * posGamma/(posGamma-negGamma))
zeroGamma = zeroGamma[0]

In [55]:
# CÉLULA 22
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(title=chartTitle, xaxis_title='Index Price', yaxis_title='Gamma Exposure ($ billions/1% move)')
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))

fig.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalGamma),
        x1=spotPrice,
        y1=max(totalGamma),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

fig.add_shape(
    dict(
        type="line",
        x0=zeroGamma,
        y0=min(totalGamma),
        x1=zeroGamma,
        y1=max(totalGamma),
        line=dict(color="green", width=1.5),
        name="Gamma Flip: " + str("{:,.0f}".format(zeroGamma))
    )
)

fig.update_xaxes(range=[fromStrike, toStrike])
fig.update_yaxes(range=[min(totalGamma), max(totalGamma)])

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[min(totalGamma), min(totalGamma), min(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="red",
        opacity=0.1,
        showlegend=False,
        name="Negative Gamma"
    )
)

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[max(totalGamma), max(totalGamma), max(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="green",
        opacity=0.1,
        showlegend=False,
        name="Positive Gamma"
    )
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1400,
    height=700
)

fig.show()

In [56]:
# CÉLULA 23
# DADOS CHART 3
# Gamma Flip: The value of the 'All Expiries' gamma profile at the Spot Price.
gamma_flip_value = np.interp(spotPrice, levels, totalGamma)
print(f"Gamma Flip (All Expiries): {gamma_flip_value:.4f} at strike {spotPrice:.0f}")

# Vol Trigger: The zero-crossing point of the 'All Expiries' line.
# This is already correctly calculated as zeroGamma.
print(f"Vol Trigger (All Expiries): {zeroGamma:.0f}")

# Max Gamma Positivo: The highest point on the 'All Expiries' line.
max_gamma_positive_value = np.max(totalGamma)
max_gamma_positive_index = np.argmax(totalGamma)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (All Expiries): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")

# Min Gamma Negativo: The lowest point on the 'All Expiries' line.
min_gamma_negative_value = np.min(totalGamma)
min_gamma_negative_index = np.argmin(totalGamma)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (All Expiries): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

Gamma Flip (All Expiries): 0.0004 at strike 26175
Vol Trigger (All Expiries): 25981
Max Gamma Positivo (All Expiries): 0.0012 at strike 26974
Min Gamma Negativo (All Expiries): -0.0042 at strike 23957


### DELTA

In [57]:
# ================================================================================
# ### DELTA (CHART 4,5 E 6)
# ================================================================================


# CÉLULA 25
# ---=== CALCULATE SPOT DELTA ===---
# Delta Exposure = Unit Delta * Open Interest * Contract Size * Spot Price
df['CallDEX'] = df['CallDelta'] * df['CallOpenInt'] * 100 * spotPrice
df['PutDEX'] = df['PutDelta'] * df['PutOpenInt'] * 100 * spotPrice

# Total Delta considers the sign of delta for calls and puts
df['TotalDelta'] = (df.CallDEX + df.PutDEX) / 10**6 # Converting to millions for better scaling

dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes_delta = dfAgg_delta.index.values

In [58]:
# ---=== CALCULATE DELTA PROFILE ===---
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

    exNxt_delta = df.loc[df['ExpirationDate'] != nextExpiry]
    totalDeltaExNext.append(exNxt_delta['callDeltaEx'].sum() + exNxt_delta['putDeltaEx'].sum())

    exFri_delta = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalDeltaExFri.append(exFri_delta['callDeltaEx'].sum() + exFri_delta['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions
totalDeltaExNext = np.array(totalDeltaExNext) / 10**6 # Converting to millions
totalDeltaExFri = np.array(totalDeltaExFri) / 10**6 # Converting to millions

# Find Delta Flip Point
zeroCrossIdx_delta = np.where(np.diff(np.sign(totalDelta)))[0]

# Handle the case where there is no zero cross
if zeroCrossIdx_delta.size > 0:
    negDelta = totalDelta[zeroCrossIdx_delta]
    posDelta = totalDelta[zeroCrossIdx_delta+1]
    negStrike_delta = levels_delta[zeroCrossIdx_delta]
    posStrike_delta = levels_delta[zeroCrossIdx_delta+1]

    zeroDelta = posStrike_delta - ((posStrike_delta - negStrike_delta) * posDelta/(posDelta-negDelta))
    # Keep zeroDelta as a single value if there's a cross, otherwise set to None or a default
    zeroDelta = zeroDelta[0] if zeroDelta.size > 0 else None
else:
    zeroDelta = None # Set zeroDelta to None if no zero cross is found

In [59]:
# CÉLULA 26
# Chart 4: Absolute Delta Exposure
# define os dados
x_data_delta = strikes_delta
y_data_delta = dfAgg_delta['TotalDelta'].to_numpy()

# cria um gráfico de barras
fig_delta4 = go.Figure(
    go.Bar(
        x=x_data_delta,
        y=y_data_delta,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Delta Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig_delta4.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data_delta),
    x1=spotPrice,
    y1=max(y_data_delta),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig_delta4.update_layout(
    title={
        'text': f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Delta Exposure ($ millions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta4.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig_delta4.show()


In [60]:
# CÉLULA 27
# --- DADOS DO CHART 4 (Delta Exposure) ---
print("="*80)
print("📊 DADOS DO CHART 4 (Delta Exposure)")
print("="*80)

# Requires dfAgg_delta from cell 5f01c222
# Requires totalDelta from cell 6143f354
# Requires zeroDelta from cell 9e43625b
# Requires df from cell I3o4YVMQogB_

dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Get the top 9 smallest delta exposure values (most negative)
smallest_delta_exposure = dfAgg_delta_sorted.head(9)
print("\nTop 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Get the top 9 largest delta exposure values (most positive)
largest_delta_exposure = dfAgg_delta_sorted.tail(9)
print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for index, row in largest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: C, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Add the Zero Delta (Delta Flip) point
# zeroDelta is calculated in cell 9e43625b
if zeroDelta is not None:
    print(f"\nZero Delta (Delta Flip): {zeroDelta:.0f}")
else:
    print("\nZero Delta (Delta Flip): Não encontrado")

# Print the total delta exposure
print(f"\nTotal Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% SPX Move")


# --- Strikes com Maior Volume e Open Interest (Geral) ---
# Referencing variables already calculated in cell e8b1a91b

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
# largest_call_vol_strike is from cell e8b1a91b
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
# largest_put_vol_strike is from cell e8b1a91b
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
# largest_call_oi_strike is from cell e8b1a91b
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
# largest_put_oi_strike is from cell e8b1a91b
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")


📊 DADOS DO CHART 4 (Delta Exposure)

Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):
  Strike 27400: Type: P, Gamma: -0.0000 Bn, Delta: -1.7706 M
  Strike 27500: Type: P, Gamma: -0.0000 Bn, Delta: -1.7972 M
  Strike 27750: Type: P, Gamma: -0.0001 Bn, Delta: -1.9050 M
  Strike 28000: Type: P, Gamma: -0.0001 Bn, Delta: -1.9415 M
  Strike 28250: Type: P, Gamma: -0.0001 Bn, Delta: -1.9851 M
  Strike 28500: Type: P, Gamma: -0.0001 Bn, Delta: -2.0211 M
  Strike 28750: Type: P, Gamma: -0.0001 Bn, Delta: -2.0486 M
  Strike 29000: Type: P, Gamma: -0.0001 Bn, Delta: -2.0724 M
  Strike 29500: Type: P, Gamma: -0.0001 Bn, Delta: -2.1119 M

Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):
  Strike 24950: Type: C, Gamma: -0.0001 Bn, Delta: 2.4465 M
  Strike 24925: Type: C, Gamma: -0.0001 Bn, Delta: 2.4367 M
  Strike 24975: Type: C, Gamma: -0.0001 Bn, Delta: 2.4275 M
  Strike 25025: Type: C, Gamma: -0.0001 Bn, Delta: 2.4002 M
  Strike 25050: Type: C, Gamma: -0.0001 Bn, Del

In [61]:
# CÉLULA 28
# Chart 5: Absolute Delta Exposure by Calls and Puts
fig_delta5 = go.Figure()
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['CallDEX'].to_numpy() / 10**6, width=6, name="Call Delta")
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['PutDEX'].to_numpy() / 10**6, width=6, name="Put Delta")
fig_delta5.update_xaxes(range=[fromStrike, toStrike])
chartTitle_delta5 = "Total Delta: $" + str("{:.2f}".format(df['TotalDelta'].sum())) + " Million per 1% SPX Move"
fig_delta5.update_layout(title_text=chartTitle_delta5, title_font=dict(size=20, family="Arial Black"))
fig_delta5.update_xaxes(title_text="Strike")
fig_delta5.update_yaxes(title_text="Spot Delta Exposure ($ millions/1% move)")
fig_delta5.add_shape(dict(type="line", x0=spotPrice, y0=min(dfAgg_delta['PutDEX'].to_numpy() / 10**6), x1=spotPrice, y1=max(dfAgg_delta['CallDEX'].to_numpy() / 10**6), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta5.update_layout(
    width=1750,
    height=800
)

fig_delta5.show()


In [62]:
# CÉLULA 29
# ---=== CALCULATE DELTA PROFILE ===---
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

    exNxt_delta = df.loc[df['ExpirationDate'] != nextExpiry]
    totalDeltaExNext.append(exNxt_delta['callDeltaEx'].sum() + exNxt_delta['putDeltaEx'].sum())

    exFri_delta = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalDeltaExFri.append(exFri_delta['callDeltaEx'].sum() + exFri_delta['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions
totalDeltaExNext = np.array(totalDeltaExNext) / 10**6 # Converting to millions
totalDeltaExFri = np.array(totalDeltaExFri) / 10**6 # Converting to millions


In [63]:
# CÉLULA 30
# Find Delta Flip Point
zeroCrossIdx_delta = np.where(np.diff(np.sign(totalDelta)))[0]

# Handle the case where there is no zero cross
if zeroCrossIdx_delta.size > 0:
    negDelta = totalDelta[zeroCrossIdx_delta]
    posDelta = totalDelta[zeroCrossIdx_delta+1]
    negStrike_delta = levels_delta[zeroCrossIdx_delta]
    posStrike_delta = levels_delta[zeroCrossIdx_delta+1]

    zeroDelta = posStrike_delta - ((posStrike_delta - negStrike_delta) * posDelta/(posDelta-negDelta))
    # Keep zeroDelta as a single value if there's a cross, otherwise set to None or a default
    zeroDelta = zeroDelta[0] if zeroDelta.size > 0 else None
else:
    zeroDelta = None # Set zeroDelta to None if no zero cross is found

In [77]:
# CÉLULA 31
# Chart 6: Delta Exposure Profile
fig_delta6 = go.Figure()

fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDelta, mode='lines', name='All Expiries'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExNext, mode='lines', name='Ex-Next Expiry'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle_delta6 = "Delta Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig_delta6.update_layout(title=chartTitle_delta6, xaxis_title='Index Price', yaxis_title='Delta Exposure ($ millions/1% move)')
fig_delta6.update_layout(title_text=chartTitle_delta6, title_font=dict(size=20, family="Arial Black"))

fig_delta6.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalDelta),
        x1=spotPrice,
        y1=max(totalDelta),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

# Add the Delta Flip line only if zeroDelta is not None
if zeroDelta is not None:
    fig_delta6.add_shape(
        dict(
            type="line",
            x0=zeroDelta,
            y0=min(totalDelta),
            x1=zeroDelta,
            y1=max(totalDelta),
            line=dict(color="green", width=1.5),
            # Format zeroDelta as a scalar
            name="Delta Flip (Zero Cross): " + str("{:,.0f}".format(float(zeroDelta)))
        )
    )

# Add marker and label for Delta Flip (Delta at Spot Price)
delta_flip_value_at_spot = np.interp(spotPrice, levels_delta, totalDelta)
fig_delta6.add_trace(
    go.Scatter(
        x=[spotPrice],
        y=[delta_flip_value_at_spot],
        mode='markers+text',
        marker=dict(color='purple', size=10),
        text=[f'Delta Flip ({spotPrice:.0f}, {delta_flip_value_at_spot:.2f}M)'],
        textposition='top center',
        showlegend=False,
        name='Delta Flip (Spot)'
    )
)


fig_delta6.update_xaxes(range=[fromStrike, toStrike])
fig_delta6.update_yaxes(range=[min(totalDelta), max(totalDelta)])

# Adding shaded areas for positive and negative delta
# Adjust shaded areas to account for potential None zeroDelta
if zeroDelta is not None:
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[min(totalDelta), min(totalDelta), min(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="red",
            opacity=0.1,
            showlegend=False,
            name="Negative Delta"
        )
    )

    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[max(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="green",
            opacity=0.1,
            showlegend=False,
            name="Positive Delta"
        )
    )
else:
     # If no zeroDelta, the entire range is either positive or negative
    fill_color = 'green' if totalDelta[0] >= 0 else 'red'
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, toStrike, toStrike, fromStrike],
            y=[min(totalDelta), min(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor=fill_color,
            opacity=0.1,
            showlegend=False,
            name="Delta Region"
        )
    )


# DEFINIR O TAMANHO DO GRAFICO EM PIXELS
fig_delta6.update_layout(
    width=1400,
    height=700
)

fig_delta6.show()

In [65]:
# CÉLULA 32
# Consolidating results from CHART 1, CHART 2, and CHART 3 - ALL DTE



print("="*80)
print("📊 RESUMO CONSOLIDADO DOS DADOS")
print("="*80)

# --- DADOS DO CHART 1 ---
print("\n--- DADOS DO CHART 1 (Gamma Exposure) ---")
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the top 9 smallest gamma values (most negative)
smallest_gamma = dfAgg_sorted.head(9)
print("Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Get the top 9 largest gamma values (most positive)
largest_gamma = dfAgg_sorted.tail(9)
print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
for index, row in largest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Add the Zero Gamma (Gamma Flip) point - Based on user's definition, this is the gamma value at Spot Price
gamma_flip_value = np.interp(spotPrice, levels, totalGamma)
print(f"\nGamma Flip (All Expiries): {gamma_flip_value:.4f} at strike {spotPrice:.0f}")

# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

# --- Dados de Open Interest e Volume para Call/Put Walls (from cell e8b1a91b) ---
print("\n--- Dados de Open Interest e Volume ---")

# Find the strike with the highest Call Open Interest and its value
call_oi_wall_strike = df.loc[df['CallOpenInt'].idxmax()]['StrikePrice']
call_oi_wall_value = df['CallOpenInt'].max()

# Find the strike with the highest Put Open Interest and its value
put_oi_wall_strike = df.loc[df['PutOpenInt'].idxmax()]['StrikePrice']
put_oi_wall_value = df['PutOpenInt'].max()

# Find the strike with the highest Call Volume and its value
call_vol_wall_strike = df.loc[df['CallVol'].idxmax()]['StrikePrice']
call_vol_wall_value = df['CallVol'].max()

# Find the strike with the highest Put Volume and its value
put_vol_wall_strike = df.loc[df['PutVol'].idxmax()]['StrikePrice']
put_vol_wall_value = df['PutVol'].max()

# Find the top 5 strikes with the highest Call Open Interest
top5_call_oi = df.nlargest(5, 'CallOpenInt')[['StrikePrice', 'CallOpenInt']]

# Find the top 5 strikes with the highest Put Open Interest
top5_put_oi = df.nlargest(5, 'PutOpenInt')[['StrikePrice', 'PutOpenInt']]

print("\nParedes por Open Interest:")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")

print("\nParedes por Volume:")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\nTop 5 Calls por Open Interest:")
for index, row in top5_call_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")

print("\nTop 5 Puts por Open Interest:")
for index, row in top5_put_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")

# --- DADOS CHART 2 (GEX Levels) ---
print("\n--- DADOS CHART 2 (GEX Levels) ---")
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

# --- DADOS CHART 3 (Gamma Profile and Delta Flip) ---
print("\n--- DADOS CHART 3 (Gamma Profile & Delta Flip) ---")
# Gamma Flip (based on user's definition: value at Spot Price for the All Expiries line)
gamma_flip_value = np.interp(spotPrice, levels, totalGamma)
print(f"Gamma Flip (All Expiries): {gamma_flip_value:.4f} at strike {spotPrice:.0f}")


# Vol Trigger (based on user's definition: zero-crossing strike of the All Expiries line)
# This is the zeroGamma value calculated earlier.
print(f"Vol Trigger (All Expiries): {zeroGamma:.0f}")

# Max Gamma Positivo (All Expiries)
max_gamma_positive_value = np.max(totalGamma)
max_gamma_positive_index = np.argmax(totalGamma)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (All Expiries): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")

# Min Gamma Negativo (All Expiries)
min_gamma_negative_value = np.min(totalGamma)
min_gamma_negative_index = np.argmin(totalGamma)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (All Expiries): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

# Delta Flip (Estrutura) from Chart 3 (based on Delta at Spot Price for Ex-Next Monthly Expiry line)
delta_at_spot_exfri = np.interp(spotPrice, levels_delta, totalDeltaExFri)
print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_at_spot_exfri:.4f} Million at strike {spotPrice:.0f}")

📊 RESUMO CONSOLIDADO DOS DADOS

--- DADOS DO CHART 1 (Gamma Exposure) ---
Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):
  Strike 25150: Type: P, Delta: 2.3507 M, Gamma: -0.0001 Bn
  Strike 25025: Type: P, Delta: 2.4002 M, Gamma: -0.0001 Bn
  Strike 25050: Type: P, Delta: 2.3872 M, Gamma: -0.0001 Bn
  Strike 25075: Type: P, Delta: 2.3769 M, Gamma: -0.0001 Bn
  Strike 25250: Type: P, Delta: 2.1064 M, Gamma: -0.0001 Bn
  Strike 25100: Type: P, Delta: 2.3618 M, Gamma: -0.0001 Bn
  Strike 25225: Type: P, Delta: 2.1433 M, Gamma: -0.0001 Bn
  Strike 25200: Type: P, Delta: 2.1316 M, Gamma: -0.0001 Bn
  Strike 25125: Type: P, Delta: 2.2241 M, Gamma: -0.0001 Bn

Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):
  Strike 26375: Type: C, Delta: -0.4339 M, Gamma: 0.0003 Bn
  Strike 26075: Type: C, Delta: 0.4615 M, Gamma: 0.0002 Bn
  Strike 26450: Type: C, Delta: -0.5824 M, Gamma: 0.0002 Bn
  Strike 25950: Type: C, Delta: 1.2336 M, Gamma: 0.0002 Bn
  Strike 26050: Type:

In [88]:
# CÉLULA 33
# Consolidating results from CHART 4, CHART 5, and CHART 6
print("--- DADOS CHART 4 ---")
# Requires dfAgg_delta from cell 5f01c222
# Requires df from cell I3o4YVMQogB_
# Requires zeroDelta from cell 9e43625b
# Requires largest_call_vol_strike, largest_put_vol_strike, largest_call_oi_strike, largest_put_oi_strike from cell e8b1a91b


dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Get the top 9 smallest delta exposure values (most negative)
smallest_delta_exposure = dfAgg_delta_sorted.head(9)
print("Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Get the top 9 largest delta exposure values (most positive)
largest_delta_exposure = dfAgg_delta_sorted.tail(9)
print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for index, row in largest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: C, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Add the Zero Delta (Delta Flip) point (based on zero cross)
if zeroDelta is not None:
    print(f"\nZero Delta (Delta Flip - Zero Cross): {zeroDelta:.0f}")
else:
    print("\nZero Delta (Delta Flip - Zero Cross): Não encontrado")


# Print the total delta exposure
print(f"\nTotal Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

# --- Strikes com Maior Volume e Open Interest (Geral) ---
# Referencing variables already calculated in cell e8b1a91b

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
# largest_call_vol_strike is from cell e8b1a91b
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
# largest_put_vol_strike is from cell e8b1a91b
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
# largest_call_oi_strike is from cell e8b1a91b
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
# largest_put_oi_strike is from cell e8b1a91b
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")


print("\n--- DADOS CHART 5 ---")
# DATA FOR CHART 5 (Absolute Delta Exposure by Calls and Puts)
# Calculate the absolute sum of Call and Put Delta Exposure for each strike
dfAgg_delta['AbsoluteTotalDEX'] = dfAgg_delta['CallDEX'].abs() + dfAgg_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure
dfAgg_delta_sorted_dex = dfAgg_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX
dex_levels = dfAgg_delta_sorted_dex.head(6)

print("Top 6 DEX Levels (based on sum of absolute Call and Put Delta Exposure):")
for i in range(len(dex_levels)):
    strike_dex = dex_levels.index[i]
    call_dex = dex_levels.iloc[i]['CallDEX'] / 10**6
    put_dex = dex_levels.iloc[i]['PutDEX'] / 10**6
    print(f"  DEX Level {i+1}: Strike {strike_dex:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")

print("\n--- DADOS CHART 6 ---")
# DATA FOR CHART 6 (Delta Exposure Profile)
# Find Delta Flip (first value above the central green line) for Ex-Next Monthly Expiry Delta Profile
# This assumes totalDeltaExFri is the data for the 'Ex-Next Monthly Expiry' line
# delta_flip_index_exfri = np.where(totalDeltaExFri > 0)[0][0] if np.any(totalDeltaExFri > 0) else None
# if delta_flip_index_exfri is not None:
#     delta_flip_strike_exfri = levels_delta[delta_flip_index_exfri]
#     delta_flip_value_exfri = totalDeltaExFri[delta_flip_index_exfri]
#     print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_flip_value_exfri:.4f} at strike {delta_flip_strike_exfri:.0f}")
# else:
#     print("No Delta Flip found (Ex-Next Monthly Expiry).")

# Delta Vol Trigger (value at zero delta cross) for Ex-Next Monthly Expiry
# Use the zeroDelta calculated earlier for the Delta Flip point
# if zeroDelta is not None:
#     delta_vol_trigger_value_at_flip_exfri = np.interp(zeroDelta, levels_delta, totalDeltaExFri)
#     print(f"Delta Vol Trigger (Delta Flip Point, Ex-Next Monthly Expiry): {delta_vol_trigger_value_at_flip_exfri:.4f} at strike {zeroDelta:.0f}")
# else:
#     print("Delta Vol Trigger not found (no Delta Flip point).")

# Max Positive Delta (Ex-Next Monthly Expiry)
max_delta_positive_value_exfri = np.max(totalDeltaExFri)
max_delta_positive_index_exfri = np.argmax(totalDeltaExFri)
max_delta_positive_strike_exfri = levels_delta[max_delta_positive_index_exfri]
print(f"Max Delta Positivo (Ex-Next Monthly Expiry): {max_delta_positive_value_exfri:.4f} at strike {max_delta_positive_strike_exfri:.0f}")

# Min Negative Delta (Ex-Next Monthly Expiry)
min_delta_negative_value_exfri = np.min(totalDeltaExFri)
min_delta_negative_index_exfri = np.argmin(totalDeltaExFri)
min_delta_negative_strike_exfri = levels_delta[min_delta_negative_index_exfri]
print(f"Min Delta Negativo (Ex-Next Monthly Expiry): {min_delta_negative_value_exfri:.4f} at strike {min_delta_negative_strike_exfri:.0f}")

# Find the Delta Exposure value at the Spot Price for the Ex-Next Monthly Expiry line and label as Delta Flip
# delta_at_spot_exfri is already calculated above
print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_at_spot_exfri:.4f} Million at strike {spotPrice:.0f}")


--- DADOS CHART 4 ---
Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):
  Strike 27400: Type: P, Gamma: -0.0000 Bn, Delta: -1.7706 M
  Strike 27500: Type: P, Gamma: -0.0000 Bn, Delta: -1.7972 M
  Strike 27750: Type: P, Gamma: -0.0001 Bn, Delta: -1.9050 M
  Strike 28000: Type: P, Gamma: -0.0001 Bn, Delta: -1.9415 M
  Strike 28250: Type: P, Gamma: -0.0001 Bn, Delta: -1.9851 M
  Strike 28500: Type: P, Gamma: -0.0001 Bn, Delta: -2.0211 M
  Strike 28750: Type: P, Gamma: -0.0001 Bn, Delta: -2.0486 M
  Strike 29000: Type: P, Gamma: -0.0001 Bn, Delta: -2.0724 M
  Strike 29500: Type: P, Gamma: -0.0001 Bn, Delta: -2.1119 M

Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):
  Strike 24950: Type: C, Gamma: -0.0001 Bn, Delta: 2.4465 M
  Strike 24925: Type: C, Gamma: -0.0001 Bn, Delta: 2.4367 M
  Strike 24975: Type: C, Gamma: -0.0001 Bn, Delta: 2.4275 M
  Strike 25025: Type: C, Gamma: -0.0001 Bn, Delta: 2.4002 M
  Strike 25050: Type: C, Gamma: -0.0001 Bn, Delta: 2.3872 M
  

### TRADING VIEW

In [89]:
# ==================== CÉLULA SIMPLIFICADA PARA TRADING VIEW - GAMMA ====================
# Esta célula gera uma string de dados reduzida com foco em Gamma para o TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FOCUS (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (GAMMA FOCUS) ====================
# Referencing variables calculated in preceding cells

# Dados gerais (including Spot Price and Date for context)
spot_price = spotPrice # from cell 12QrnqhXqLJe
update_date = todayDate.strftime('%Y-%m-%d %H:%M') # from cell 12QrnqhXqLJe

# GAMMA EXPOSURE TOP 5 STRIKES POR GAMMA EXPOSURE NEGATIVO
dfAgg_sorted_gamma = dfAgg.sort_values(by='TotalGamma') # dfAgg from cell HaWB9gJNrVxk
smallest_gamma_5 = dfAgg_sorted_gamma.head(5)
gamma_neg_data = []
for i, (index, row) in enumerate(smallest_gamma_5.iloc[::-1].iterrows()):
    gamma_neg_data.append({'name': f'NegG{i+1}', 'strike': index, 'gamma': row['TotalGamma']})

# GAMMA EXPOSURE TOP 5 STRIKES POR GAMMA EXPOSURE POSITIVO
largest_gamma_5 = dfAgg_sorted_gamma.tail(5)
gamma_pos_data = []
for i, (index, row) in enumerate(largest_gamma_5.iloc[::-1].iterrows()):
     gamma_pos_data.append({'name': f'PosG{i+1}', 'strike': index, 'gamma': row['TotalGamma']})

# PAREDES POR OPEN INTEREST (from cell DZaTkhZ0rwkL)
call_oi_wall_strike = call_oi_wall_strike
call_oi_wall_value = call_oi_wall_value
put_oi_wall_strike = put_oi_wall_strike
put_oi_wall_value = put_oi_wall_value

# PAREDES POR VOLUME (from cell DZaTkhZ0rwkL)
call_vol_wall_strike = call_vol_wall_strike
call_vol_wall_value = call_vol_wall_value
put_vol_wall_strike = put_vol_wall_strike
put_vol_wall_value = put_vol_wall_value

# TOP 5 CALLS POR OPEN INTEREST (from cell DZaTkhZ0rwkL)
top5_call_oi_data = top5_call_oi.to_dict('records')

# TOP 5 PUTS POR OPEN INTEREST (from cell DZaTkhZ0rwkL)
top5_put_oi_data = top5_put_oi.to_dict('records')

# TOP GEX LEVELS (from cell K79nEo18sCWz)
gex_levels = dfAgg_sorted_gex.head(6) # gex_levels is already calculated and sorted in cell K79nEo18sCWz
gex_levels_data = []
for i in range(min(6, len(gex_levels))): # Take min of 6 or available levels
    gex_levels_data.append({
        'name': f'GEX{i+1}',
        'strike': gex_levels.index[i],
        'call_gex': gex_levels.iloc[i]['CallGEX'] / 10**9, # Convert to billions
        'put_gex': gex_levels.iloc[i]['PutGEX'] / 10**9   # Convert to billions
    })

# MAX GAMMA POS (All Expiries - from cell 1lFAPvoRsWnS)
max_gamma_positive_value = np.max(totalGamma) # from cell 1lFAPvoRsWnS
max_gamma_positive_strike = levels[np.argmax(totalGamma)] # from cell 1lFAPvoRsWnS

# MIN GAMMA NEG (All Expiries - from cell 1lFAPvoRsWnS)
min_gamma_negative_value = np.min(totalGamma) # from cell 1lFAPvoRsWnS
min_gamma_negative_strike = levels[np.argmin(totalGamma)] # from cell 1lFAPvarsWnS


# ==================== GERAÇÃO DA LINHA ÚNICA (GAMMA FOCUS) ====================

# Criar a string com todos os dados separados por vírgula em formato key=value
data_string_gamma_focus = ""

# Top 5 Neg Gamma (5 strikes * (name=value, name=value))
for item in gamma_neg_data:
    data_string_gamma_focus += f"{item['name']}={item['gamma']:.4f},{item['name']}_S={item['strike']:.0f},"
# Top 5 Pos Gamma (5 strikes * (name=value, name=value))
for item in gamma_pos_data:
     data_string_gamma_focus += f"{item['name']}={item['gamma']:.4f},{item['name']}_S={item['strike']:.0f},"
# OI Walls (2 walls * (name=value, name=value))
data_string_gamma_focus += f"CallOIWall_V={call_oi_wall_value:.0f},CallOIWall_S={call_oi_wall_strike:.0f},"
data_string_gamma_focus += f"PutOIWall_V={put_oi_wall_value:.0f},PutOIWall_S={put_oi_wall_strike:.0f},"
# Vol Walls (2 walls * (name=value, name=value))
data_string_gamma_focus += f"CallVolWall_V={call_vol_wall_value:.0f},CallVolWall_S={call_vol_wall_strike:.0f},"
data_string_gamma_focus += f"PutVolWall_V={put_vol_wall_value:.0f},PutVolWall_S={put_vol_wall_strike:.0f},"
# Top 5 Calls OI (5 strikes * (name=value, name=value))
for i, item in enumerate(top5_call_oi_data):
    data_string_gamma_focus += f"Top5CallOI_S{i+1}={item['StrikePrice']:.0f},Top5CallOI_V{i+1}={item['CallOpenInt']:.0f},"
# Top 5 Puts OI (5 strikes * (name=value, name=value))
for i, item in enumerate(top5_put_oi_data):
    data_string_gamma_focus += f"Top5PutOI_S{i+1}={item['StrikePrice']:.0f},Top5PutOI_V{i+1}={item['PutOpenInt']:.0f},"
# Top 6 GEX Levels (6 levels * (name=value, name=value, name=value))
for item in gex_levels_data:
    data_string_gamma_focus += f"{item['name']}_S={item['strike']:.0f},{item['name']}_CallGEX={item['call_gex']:.4f},{item['name']}_PutGEX={item['put_gex']:.4f},"
# Max Gamma Pos (Value & Strike)
data_string_gamma_focus += f"MaxPosGamma_V={max_gamma_positive_value:.4f},MaxPosGamma_S={max_gamma_positive_strike:.0f},"
# Min Gamma Neg (Value & Strike)
data_string_gamma_focus += f"MinNegGamma_V={min_gamma_negative_value:.4f},MinNegGamma_S={min_gamma_negative_strike:.0f},"


# General Data (Spot Price + date) - Moved to the end
# Ensure the last value does not have a trailing comma before adding general data
data_string_gamma_focus = data_string_gamma_focus.rstrip(',')
data_string_gamma_focus += f",SpotPrice={spot_price:.2f},UpdateDate={update_date}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:\n")
print("="*80)
print(data_string_gamma_focus)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR:\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador que você está usando.")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' (ou similar) no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (GAMMA FOCUS - SIMPLIFICADA):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📅 DATA: {update_date}")

print("\n--- GAMMA EXPOSURE ---")
print("\n🔴 Top 5 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
for item in gamma_neg_data:
    print(f"  Strike {item['strike']:.0f}: Gamma: {item['gamma']:.4f} Bn")

print("\n🟢 Top 5 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
for item in gamma_pos_data:
    print(f"  Strike {item['strike']:.0f}: Gamma: {item['gamma']:.4f} Bn")

print("\n📊 Paredes por Open Interest:")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")

print("\n📊 Paredes por Volume:")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\n📈 Top 5 Calls por Open Interest:")
for item in top5_call_oi_data:
    print(f"  Strike {item['StrikePrice']:.0f}: {item['CallOpenInt']:.0f}")

print("\n📉 Top 5 Puts por Open Interest:")
for item in top5_put_oi_data:
    print(f"  Strike {item['StrikePrice']:.0f}: {item['PutOpenInt']:.0f}")

print("\n--- GEX LEVELS ---")
print("\n💎 Top 6 GEX Levels:")
for i, item in enumerate(gex_levels_data):
    net_gex = item['call_gex'] + item['put_gex']
    print(f"   {i+1}. Strike {item['strike']:.0f}: Net GEX = {net_gex:.2f} Bn (Call GEX: {item['call_gex']:.4f} Bn, Put GEX: {item['put_gex']:.4f} Bn)")

print("\n--- GAMMA PROFILE POINTS (ALL EXPIRIES) ---")
print(f"   • Max Gamma Pos: {max_gamma_positive_strike:.0f} (γ: {max_gamma_positive_value:.4f} Bn)")
print(f"   • Min Gamma Neg: {min_gamma_negative_strike:.0f} (γ: {min_gamma_negative_value:.4f} Bn)")


print("\n" + "="*80)
print("✅ DADOS SIMPLIFICADOS (GAMMA FOCUS) PRONTOS PARA USAR!")
print("="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FOCUS (VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:

NegG1=-0.0001,NegG1_S=25250,NegG2=-0.0001,NegG2_S=25100,NegG3=-0.0001,NegG3_S=25225,NegG4=-0.0001,NegG4_S=25200,NegG5=-0.0001,NegG5_S=25125,PosG1=0.0003,PosG1_S=26375,PosG2=0.0002,PosG2_S=26075,PosG3=0.0002,PosG3_S=26450,PosG4=0.0002,PosG4_S=25950,PosG5=0.0002,PosG5_S=26050,CallOIWall_V=1,CallOIWall_S=24925,PutOIWall_V=1,PutOIWall_S=24925,CallVolWall_V=0,CallVolWall_S=24925,PutVolWall_V=0,PutVolWall_S=24925,Top5CallOI_S1=24925,Top5CallOI_V1=1,Top5CallOI_S2=24950,Top5CallOI_V2=1,Top5CallOI_S3=24975,Top5CallOI_V3=1,Top5CallOI_S4=25000,Top5CallOI_V4=1,Top5CallOI_S5=25025,Top5CallOI_V5=1,Top5PutOI_S1=24925,Top5PutOI_V1=1,Top5PutOI_S2=24950,Top5PutOI_V2=1,Top5PutOI_S3=24975,Top5PutOI_V3=1,Top5PutOI_S4=25000,Top5PutOI_V4=1,Top5PutOI_S5=25025,Top5PutOI_V5=1,GEX1_S=26050,GEX1_CallGEX=0.0006,GEX1_PutGEX=-0.0004,GEX2_S=26225,GEX2_CallGEX=0.0006,GEX2_PutGEX=-0.0004,GEX3_S

In [80]:
# ==================== CÉLULA SIMPLIFICADA PARA TRADING VIEW - DELTA ====================
# Esta célula gera uma string de dados reduzida com foco em Delta para o TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA FOCUS (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (DELTA FOCUS) ====================
# Referencing variables calculated in preceding cells

# Dados gerais (including Spot Price and Date for context)
spot_price = spotPrice # from cell 12QrnqhXqLJe
update_date = todayDate.strftime('%Y-%m-%d %H:%M') # from cell 12QrnqhXqLJe

# DELTA EXPOSURE TOP 5 STRIKES POR DELTA EXPOSURE NEGATIVO
dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta') # dfAgg_delta from cell hT_lACSWsbNw
smallest_delta_exposure_5 = dfAgg_delta_sorted.head(5)
delta_neg_data = []
for i, (index, row) in enumerate(smallest_delta_exposure_5.iloc[::-1].iterrows()):
    delta_neg_data.append({'name': f'NegD{i+1}', 'strike': index, 'delta': row['TotalDelta']})


# DELTA EXPOSURE TOP 5 STRIKES POR DELTA EXPOSURE POSITIVO
largest_delta_exposure_5 = dfAgg_delta_sorted.tail(5)
delta_pos_data = []
for i, (index, row) in enumerate(largest_delta_exposure_5.iloc[::-1].iterrows()):
     delta_pos_data.append({'name': f'PosD{i+1}', 'strike': index, 'delta': row['TotalDelta']})


# TOP 5 DEX LEVELS (from cell 4YcrtF6VtKE5)
# dex_levels is from cell 4YcrtF6VtKE5 - ensure it's calculated there or recalculate if needed
# Recalculate dex_levels to be safe, as the previous cell might have been skipped
dfAgg_delta['AbsoluteTotalDEX'] = dfAgg_delta['CallDEX'].abs() + dfAgg_delta['PutDEX'].abs()
dfAgg_delta_sorted_dex = dfAgg_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)
dex_levels = dfAgg_delta_sorted_dex.head(5) # User requested Top 5 DEX levels

dex_levels_data = []
for i in range(min(5, len(dex_levels))): # Take min of 5 or available levels
    dex_levels_data.append({
        'name': f'DEX{i+1}',
        'strike': dex_levels.index[i],
        'call_dex': dex_levels.iloc[i]['CallDEX'] / 10**6, # Convert to millions
        'put_dex': dex_levels.iloc[i]['PutDEX'] / 10**6   # Convert to millions
    })

# MAX DELTA POS (Ex-Next Monthly Expiry - from cell 4YcrtF6VtKE5)
# Ensure totalDeltaExFri and levels_delta are available (calculated in cell AzJQWKqus37Q)
max_delta_positive_value_exfri = np.max(totalDeltaExFri)
max_delta_positive_strike_exfri = levels_delta[np.argmax(totalDeltaExFri)]

# MIN DELTA NEG (Ex-Next Monthly Expiry - from cell 4YcrtF6VtKE5)
min_delta_negative_value_exfri = np.min(totalDeltaExFri)
min_delta_negative_strike_exfri = levels_delta[np.argmin(totalDeltaExFri)]


# ==================== GERAÇÃO DA LINHA ÚNICA (DELTA FOCUS) ====================

# Criar a string com todos os dados separados por vírgula em formato key=value
data_string_delta_focus = ""

# Top 5 Neg Delta (5 strikes * (name=value, name=value))
for item in delta_neg_data:
    data_string_delta_focus += f"{item['name']}={item['delta']:.4f},{item['name']}_S={item['strike']:.0f},"
# Top 5 Pos Delta (5 strikes * (name=value, name=value))
for item in delta_pos_data:
     data_string_delta_focus += f"{item['name']}={item['delta']:.4f},{item['name']}_S={item['strike']:.0f},"
# Top 5 DEX Levels (5 levels * (name=value, name=value, name=value))
for item in dex_levels_data:
    data_string_delta_focus += f"{item['name']}_S={item['strike']:.0f},{item['name']}_CallDEX={item['call_dex']:.4f},{item['name']}_PutDEX={item['put_dex']:.4f},"
# Max Delta Pos (Value & Strike)
data_string_delta_focus += f"MaxPosDelta_V={max_delta_positive_value_exfri:.4f},MaxPosDelta_S={max_delta_positive_strike_exfri:.0f},"
# Min Delta Neg (Value & Strike)
data_string_delta_focus += f"MinNegDelta_V={min_delta_negative_value_exfri:.4f},MinNegDelta_S={min_delta_negative_strike_exfri:.0f},"


# General Data (Spot Price + date) - Moved to the end
# Ensure the last value does not have a trailing comma before adding general data
data_string_delta_focus = data_string_delta_focus.rstrip(',')
data_string_delta_focus += f",SpotPrice={spot_price:.2f},UpdateDate={update_date}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:\n")
print("="*80)
print(data_string_delta_focus)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR:\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador que você está usando.")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' (ou similar) no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (DELTA FOCUS - SIMPLIFICADA):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📅 DATA: {update_date}")

print("\n--- DELTA EXPOSURE ---")
print("\n🔴 Top 5 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for item in delta_neg_data:
    print(f"  Strike {item['strike']:.0f}: Delta: {item['delta']:.4f} M")

print("\n🟢 Top 5 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for item in delta_pos_data:
    print(f"  Strike {item['strike']:.0f}: Delta: {item['delta']:.4f} M")

print("\n--- DEX LEVELS ---")
print("\n💎 Top 5 DEX Levels:")
for i, item in enumerate(dex_levels_data):
    net_dex = item['call_dex'] + item['put_dex']
    print(f"   {i+1}. Strike {item['strike']:.0f}: Net DEX = {net_dex:.2f} M (Call DEX: {item['call_dex']:.4f} M, Put DEX: {item['put_dex']:.4f} M)")


print("\n--- DELTA PROFILE POINTS (EX-NEXT MONTHLY EXPIRY) ---")
print(f"   • Max Delta Pos: {max_delta_positive_strike_exfri:.0f} (Δ: {max_delta_positive_value_exfri:.4f} M)")
print(f"   • Min Delta Neg: {min_delta_negative_strike_exfri:.0f} (Δ: {min_delta_negative_value_exfri:.4f} M)")

print("\n" + "="*80)
print("✅ DADOS SIMPLIFICADOS (DELTA FOCUS) PRONTOS PARA USAR!")
print("="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA FOCUS (VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:

NegD1=-1.9851,NegD1_S=28250,NegD2=-2.0211,NegD2_S=28500,NegD3=-2.0486,NegD3_S=28750,NegD4=-2.0724,NegD4_S=29000,NegD5=-2.1119,NegD5_S=29500,PosD1=2.4465,PosD1_S=24950,PosD2=2.4367,PosD2_S=24925,PosD3=2.4275,PosD3_S=24975,PosD4=2.4002,PosD4_S=25025,PosD5=2.3872,PosD5_S=25050,DEX1_S=25525,DEX1_CallDEX=2.4145,DEX1_PutDEX=-0.6615,DEX2_S=25375,DEX2_CallDEX=2.5087,DEX2_PutDEX=-0.5486,DEX3_S=25450,DEX3_CallDEX=2.4673,DEX3_PutDEX=-0.5678,DEX4_S=25350,DEX4_CallDEX=2.5202,DEX4_PutDEX=-0.4972,DEX5_S=25300,DEX5_CallDEX=2.5403,DEX5_PutDEX=-0.4752,MaxPosDelta_V=0.0000,MaxPosDelta_S=20940,MinNegDelta_V=0.0000,MinNegDelta_S=20940,SpotPrice=26175.00,UpdateDate=2025-10-31 14:00

💡 COMO USAR:

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador que você está usando.
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' 

In [81]:
# ==================== CÉLULA SIMPLIFICADA PARA TRADING VIEW - PONTOS IMPORTANTES ====================
# Esta célula gera uma string de dados reduzida com foco em pontos importantes para o TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - IMPORTANT POINTS (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (IMPORTANT POINTS) ====================
# Referencing variables calculated in preceding cells

# Dados gerais (including Spot Price and Date for context)
spot_price = spotPrice # from cell 12QrnqhXqLJe
update_date = todayDate.strftime('%Y-%m-%d %H:%M') # from cell 12QrnqhXqLJe

# PAREDES POR OPEN INTEREST (from cell DZaTkhZ0rwkL)
call_oi_wall_strike = call_oi_wall_strike
call_oi_wall_value = call_oi_wall_value
put_oi_wall_strike = put_oi_wall_strike
put_oi_wall_value = put_oi_wall_value

# PAREDES POR VOLUME (from cell DZaTkhZ0rwkL)
call_vol_wall_strike = call_vol_wall_strike
call_vol_wall_value = call_vol_wall_value
put_vol_wall_strike = put_vol_wall_strike
put_vol_wall_value = put_vol_wall_value

# GAMMA FLIP (from cell 1lFAPvoRsWnS - value at Spot Price)
gamma_flip_value = np.interp(spotPrice, levels, totalGamma)

# VOL TRIGGER (from cell 1lFAPvoRsWnS - zero crossing strike)
vol_trigger_strike = zeroGamma

# MAX GAMMA POS (All Expiries - from cell 1lFAPvoRsWnS)
max_gamma_positive_value = np.max(totalGamma)
max_gamma_positive_strike = levels[np.argmax(totalGamma)]

# MIN GAMMA NEG (All Expiries - from cell 1lFAPvoRsWnS)
min_gamma_negative_value = np.min(totalGamma)
min_gamma_negative_strike = levels[np.argmin(totalGamma)]

# DELTA FLIP (ESTRUTURA) (from cell 1lFAPvoRsWnS - Delta at Spot Price for Ex-Next Monthly Expiry)
delta_flip_estrutura_strike = spotPrice
delta_flip_estrutura_value = np.interp(spotPrice, levels_delta, totalDeltaExFri)

# DELTA FLIP (OI) (from cell 4YcrtF6VtKE5 or 60dc6097 - Delta at Spot Price for Ex-Next Monthly Expiry, same as Estrutura by definition)
# Reusing delta_flip_estrutura_strike and delta_flip_estrutura_value as the definition is the same
chart6_delta_flip_oi_strike = delta_flip_estrutura_strike
chart6_delta_flip_oi_value = delta_flip_estrutura_value

# MAX DELTA POS (Ex-Next Monthly Expiry - from cell 60dc6097)
max_delta_positive_value_exfri = np.max(totalDeltaExFri)
max_delta_positive_strike_exfri = levels_delta[np.argmax(totalDeltaExFri)]

# MIN DELTA NEG (Ex-Next Monthly Expiry - from cell 60dc6097)
min_delta_negative_value_exfri = np.min(totalDeltaExFri)
min_delta_negative_strike_exfri = levels_delta[np.argmin(totalDeltaExFri)]


# ==================== GERAÇÃO DA LINHA ÚNICA (IMPORTANT POINTS) ====================

# Criar a string com todos os dados separados por vírgula em formato key=value
data_string_important_points = ""

# Call Wall (OI) (Value & Strike)
data_string_important_points += f"CallOIWall_V={call_oi_wall_value:.0f},CallOIWall_S={call_oi_wall_strike:.0f},"
# Put Wall (OI) (Value & Strike)
data_string_important_points += f"PutOIWall_V={put_oi_wall_value:.0f},PutOIWall_S={put_oi_wall_strike:.0f},"
# Call Wall (VOL) (Value & Strike)
data_string_important_points += f"CallVolWall_V={call_vol_wall_value:.0f},CallVolWall_S={call_vol_wall_strike:.0f},"
# Put Wall (VOL) (Value & Strike)
data_string_important_points += f"PutVolWall_V={put_vol_wall_value:.0f},PutVolWall_S={put_vol_wall_strike:.0f},"
# Gamma Flip (Value & Strike at Spot)
data_string_important_points += f"GammaFlip_V={gamma_flip_value:.4f},GammaFlip_S={spot_price:.0f},"
# Vol Trigger (Value at zeroGamma & Strike at zeroGamma)
data_string_important_points += f"VolTrigger_V={np.interp(zeroGamma, levels, totalGamma):.4f},VolTrigger_S={zeroGamma:.0f},"
# Max Gamma Pos (Value & Strike)
data_string_important_points += f"MaxPosGamma_V={max_gamma_positive_value:.4f},MaxPosGamma_S={max_gamma_positive_strike:.0f},"
# Min Gamma Neg (Value & Strike)
data_string_important_points += f"MinNegGamma_V={min_gamma_negative_value:.4f},MinNegGamma_S={min_gamma_negative_strike:.0f},"
# Delta Flip (Estrutura) (Value & Strike at Spot)
data_string_important_points += f"DeltaFlipEstrutura_V={delta_flip_estrutura_value:.4f},DeltaFlipEstrutura_S={delta_flip_estrutura_strike:.0f},"
# Delta Flip (OI) (Value & Strike at Spot) - Using the same value as Estrutura
data_string_important_points += f"DeltaFlipOI_V={chart6_delta_flip_oi_value:.4f},DeltaFlipOI_S={chart6_delta_flip_oi_strike:.0f},"
# Max Delta Pos (Ex-Next Monthly) (Value & Strike)
data_string_important_points += f"MaxPosDelta_V={max_delta_positive_value_exfri:.4f},MaxPosDelta_S={max_delta_positive_strike_exfri:.0f},"
# Min Delta Neg (Ex-Next Monthly) (Value & Strike)
data_string_important_points += f"MinNegDelta_V={min_delta_negative_value_exfri:.4f},MinNegDelta_S={min_delta_negative_strike_exfri:.0f},"


# General Data (Spot Price + date) - Moved to the end
# Ensure the last value does not have a trailing comma before adding general data
data_string_important_points = data_string_important_points.rstrip(',')
data_string_important_points += f",SpotPrice={spot_price:.2f},UpdateDate={update_date}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:\n")
print("="*80)
print(data_string_important_points)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR:\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador que você está usando.")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' (ou similar) no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (IMPORTANT POINTS - SIMPLIFICADA):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📅 DATA: {update_date}")

print("\n--- WALLS ---")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\n--- GAMMA POINTS ---")
print(f"  Gamma Flip (All Expiries): {spot_price:.0f} (γ: {gamma_flip_value:.4f} Bn)")
print(f"  Vol Trigger (All Expiries): {vol_trigger_strike:.0f} (γ: {np.interp(zeroGamma, levels, totalGamma):.4f} Bn)")
print(f"  Max Gamma Pos (All Expiries): {max_gamma_positive_strike:.0f} (γ: {max_gamma_positive_value:.4f} Bn)")
print(f"  Min Gamma Neg (All Expiries): {min_gamma_negative_strike:.0f} (γ: {min_gamma_negative_value:.4f} Bn)")

print("\n--- DELTA POINTS ---")
print(f"  Delta Flip (Estrutura): {delta_flip_estrutura_strike:.0f} (Δ: {delta_flip_estrutura_value:.4f} M)")
print(f"  Delta Flip (OI): {chart6_delta_flip_oi_strike:.0f} (Δ: {chart6_delta_flip_oi_value:.4f} M)")
print(f"  Max Delta Pos (Ex-Next Monthly): {max_delta_positive_strike_exfri:.0f} (Δ: {max_delta_positive_value_exfri:.4f} M)")
print(f"  Min Delta Neg (Ex-Next Monthly): {min_delta_negative_strike_exfri:.0f} (Δ: {min_delta_negative_value_exfri:.4f} M)")


print("\n" + "="*80)
print("✅ DADOS SIMPLIFICADOS (IMPORTANT POINTS) PRONTOS PARA USAR!")
print("="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - IMPORTANT POINTS (VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:

CallOIWall_V=1,CallOIWall_S=24925,PutOIWall_V=1,PutOIWall_S=24925,CallVolWall_V=0,CallVolWall_S=24925,PutVolWall_V=0,PutVolWall_S=24925,GammaFlip_V=0.0004,GammaFlip_S=26175,VolTrigger_V=0.0000,VolTrigger_S=25981,MaxPosGamma_V=0.0012,MaxPosGamma_S=26974,MinNegGamma_V=-0.0042,MinNegGamma_S=23957,DeltaFlipEstrutura_V=0.0000,DeltaFlipEstrutura_S=26175,DeltaFlipOI_V=0.0000,DeltaFlipOI_S=26175,MaxPosDelta_V=0.0000,MaxPosDelta_S=20940,MinNegDelta_V=0.0000,MinNegDelta_S=20940,SpotPrice=26175.00,UpdateDate=2025-10-31 14:00

💡 COMO USAR:

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador que você está usando.
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' (ou similar) no grupo 'Dados de Entrada'
5. ✅ COLE a linha copiada neste campo
6. ✅ Clique em 'OK'
7. ✅ PRONTO! Os dados serão atualizados autom